<a href="https://colab.research.google.com/github/Bhimashankar05/Stock_data_downloader/blob/main/app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
import streamlit as st
import yfinance as yf
import pandas as pd
import logging
from datetime import date, timedelta

# ---------------- LOGGING ----------------

In [23]:
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# ---------------- PAGE CONFIG ----------------

In [24]:
st.set_page_config(page_title="Stock Data Downloader", layout="wide")
st.title("📈 NSE / BSE Stock Data Downloader")

2026-08-12 18:47:22.328 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-12 18:47:22.330 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-12 18:47:22.486 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-08-12 18:47:22.487 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-12 18:47:22.488 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


DeltaGenerator()

In [25]:
# ---------------- SIDEBAR ----------------

In [26]:
st.sidebar.header("⚙️ Configuration")

2026-08-12 18:47:23.722 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-12 18:47:23.724 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-12 18:47:23.725 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


DeltaGenerator(_root_container=1, _parent=DeltaGenerator())

In [27]:
exchange = st.sidebar.selectbox("Select Exchange", ["NSE", "BSE"])
suffix = ".NS" if exchange == "NSE" else ".BO"

data_type = st.sidebar.selectbox(
    "Data Type",
    ["Daily", "Weekly", "Monthly", "Intraday"]
)

2026-08-12 18:47:23.908 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-12 18:47:23.910 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-12 18:47:23.911 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-12 18:47:23.912 Session state does not function when running a script without `streamlit run`
2026-08-12 18:47:23.914 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-12 18:47:23.915 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-12 18:47:23.915 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-12 18:47:23.917 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-12 18:47

In [28]:
interval_map = {
    "Daily": "1d",
    "Weekly": "1wk",
    "Monthly": "1mo"
}


In [29]:
interval = "1d"
if data_type == "Intraday":
    interval = st.sidebar.selectbox(
        "Intraday Interval",
        ["1m", "5m", "15m", "30m", "60m"]
    )

start_date = st.sidebar.date_input(
    "Start Date", date.today() - timedelta(days=30)
)
end_date = st.sidebar.date_input("End Date", date.today())

output_format = st.sidebar.selectbox("Output Format", ["CSV", "Excel"])

2026-08-12 18:47:25.219 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-12 18:47:25.221 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-12 18:47:25.223 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-12 18:47:25.224 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-12 18:47:25.226 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-12 18:47:25.228 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-12 18:47:25.230 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-12 18:47:25.234 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

# ---------------- INTRADAY NOTE ----------------

In [30]:
if data_type == "Intraday":
    st.info(
        "ℹ️ **Intraday Data Limitation (Yahoo Finance)**\n\n"
        "- **1m interval** → last **7 days only**\n"
        "- **5m / 15m / 30m / 60m** → last **60 days only**\n"
        "- Date range will be **auto-adjusted automatically**"
    )

# ---------------- SYMBOL INPUT ----------------

In [31]:
st.subheader("📌 Stock Symbols")

input_mode = st.radio(
    "Choose symbol input method",
    ["Upload CSV / Excel", "Enter symbols manually"]
)

2026-08-12 18:47:26.994 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-12 18:47:26.996 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-12 18:47:26.997 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-12 18:47:26.999 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-12 18:47:27.001 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-12 18:47:27.003 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-12 18:47:27.005 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-12 18:47:27.006 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [32]:
symbols = []

In [33]:
if input_mode == "Upload CSV / Excel":
    uploaded_file = st.file_uploader(
        "Upload file (must contain `Symbol` column)",
        type=["csv", "xlsx"]
    )

    if uploaded_file:
        if uploaded_file.name.endswith(".csv"):
            df = pd.read_csv(uploaded_file)
        else:
            df = pd.read_excel(uploaded_file)

        symbols = df["Symbol"].dropna().astype(str).unique().tolist()

else:
    symbol_text = st.text_input(
        "Enter symbols (comma-separated)",
        placeholder="SBIN, INFY, TCS, HDFCBANK"
    )

    if symbol_text:
        symbols = [s.strip().upper() for s in symbol_text.split(",") if s.strip()]

2026-08-12 18:47:29.559 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-12 18:47:29.562 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-12 18:47:29.564 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-12 18:47:29.565 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-12 18:47:29.566 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


# ---------------- FETCH BUTTON (ONLY ONCE) ----------------

In [34]:
fetch_clicked = st.button("🚀 Fetch Data")


2026-08-12 18:47:32.193 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-12 18:47:32.196 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-12 18:47:32.199 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-12 18:47:32.202 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-12 18:47:32.205 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


# ---------------- FETCH LOGIC ----------------

In [35]:
if fetch_clicked:
    if not symbols:
        st.warning("Please upload a file or enter at least one symbol.")
        st.stop()

    all_data = []
    failed_symbols = []

    # ---- Intraday date enforcement ----
    if data_type == "Intraday":
        max_days = 7 if interval == "1m" else 60
        allowed_start = end_date - timedelta(days=max_days)

        if start_date < allowed_start:
            st.warning(
                f"⚠️ Start date adjusted to last {max_days} days "
                f"due to intraday limits."
            )
            start_date = allowed_start

    progress = st.progress(0.0)

    for i, symbol in enumerate(symbols):
        try:
            ticker = yf.Ticker(f"{symbol}{suffix}")

            data = ticker.history(
                start=start_date,
                end=end_date,
                interval=interval if data_type == "Intraday" else interval_map[data_type],
                auto_adjust=False,
                prepost=False
            )

            if not data.empty:
                data = data[["Open", "High", "Low", "Close", "Volume"]]
                data["Symbol"] = symbol
                data.reset_index(inplace=True)

                # ---- Fix Date vs Datetime ----
                if "Datetime" in data.columns:
                    data.rename(columns={"Datetime": "Date"}, inplace=True)

                all_data.append(data)
            else:
                failed_symbols.append(symbol)

        except Exception as e:
            logger.error(f"{symbol} failed: {e}")
            failed_symbols.append(symbol)

        progress.progress((i + 1) / len(symbols))

    # ---------------- FINAL DATAFRAME ----------------
    if not all_data:
        st.error("❌ No data fetched. Check symbols or date range.")
        st.stop()

    final_df = pd.concat(all_data, ignore_index=True)
    final_df = final_df[
        ["Date", "Symbol", "Open", "High", "Low", "Close", "Volume"]
    ]

    # ---------------- EXCEL TIMEZONE FIX ----------------
    if pd.api.types.is_datetime64_any_dtype(final_df["Date"]):
        final_df["Date"] = final_df["Date"].dt.tz_localize(None)

    st.success("✅ Data fetched successfully")
    st.dataframe(final_df, use_container_width=True)

    # ---------------- DOWNLOAD ----------------
    if output_format == "CSV":
        csv = final_df.to_csv(index=False).encode("utf-8")
        st.download_button(
            "⬇️ Download CSV",
            csv,
            file_name="stock_data.csv",
            mime="text/csv"
        )
    else:
        with pd.ExcelWriter("stock_data.xlsx", engine="xlsxwriter") as writer:
            final_df.to_excel(writer, index=False, sheet_name="Data")

        with open("stock_data.xlsx", "rb") as f:
            st.download_button(
                "⬇️ Download Excel",
                f,
                file_name="stock_data.xlsx",
                mime="application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"
            )

    if failed_symbols:
        st.warning(f"⚠️ Failed symbols: {', '.join(failed_symbols)}")
